In [0]:
spark.sql("USE fraud")

In [0]:
suspicious_df = spark.sql("""
SELECT *
FROM fraud.suspicious_transactions
LIMIT 20
""")

suspicious_df.display()

In [0]:
transactions_json = suspicious_df.toPandas().to_json(orient="records")
print(transactions_json[:1000])

In [0]:
from openai import AzureOpenAI

# Fetch secrets from Databricks Secret Scope connected to Key Vault
api_key = dbutils.secrets.get("ai-secrets", "AZURE-OPENAI-KEY")
endpoint = dbutils.secrets.get("ai-secrets", "AZURE-OPENAI-ENDPOINT")
api_version = dbutils.secrets.get("ai-secrets", "AZURE-OPENAI-API-VERSION")
deployment = "gpt-4.1-mini"

client = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint
)

response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "system", "content": "You are a financial fraud detection expert."},
        {"role": "user", "content": "Analyze if this transaction looks suspicious: customer C22 paid 5600 in Dubai."}
    ]
)

print(response.choices[0].message.content)

In [0]:
prompt = f"""
You are a financial fraud detection AI.

Analyze the following transactions and determine if they appear suspicious.
Explain the reason for any fraud risk.

Transactions:
{transactions_json}
"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role":"system","content":"You are an expert financial fraud detection system."},
        {"role":"user","content":prompt}
    ],
    temperature=0.2
)

analysis = response.choices[0].message.content

print(analysis)

In [0]:
analysis_df = spark.createDataFrame([(analysis,)], ["fraud_analysis"])

analysis_df.write.format("delta").mode("overwrite").saveAsTable("fraud.ai_fraud_analysis")

In [0]:
spark.sql("SELECT * FROM fraud.ai_fraud_analysis").display()